# RIGHTWARE SIC — E1-X02 | Query-balanced pairwise ranking
**Protocolo experimental congelado, não validação científica.** Importar no Kaggle, CPU (Accelerator None), Internet Off, Run All. Sem ficheiros de entrada, credenciais ou APIs. Exporta `RIGHTWARE_SIC_E1_X02_EVIDENCE.zip`.

X01 (SI-MH-0086) mostrou generalização fraca e abstenção total. X02 testa novas entidades, famílias de formulação, candidatos comparáveis, características explícitas de entidade/atributo e treino pairwise. Todos os dados e rótulos são sintéticos, da mesma equipa; diagnóstico não é holdout independente. Não consultar E1.1B, A0 v0.8 ou dados privados. E1_DATA_READY continua REVIEW_REQUIRED.

In [ ]:
import os
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='1'
import json, re, math, random, hashlib, time, platform, zipfile, sys, uuid
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
SEED=20261125
rng=random.Random(SEED)
np_rng=np.random.default_rng(SEED)
K=5
# No reading from /kaggle/input, no network, no installation.
ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUT=ROOT/('SIC_E1_X02_'+uuid.uuid4().hex[:10]); OUT.mkdir()
PROTOCOL={'experiment':'E1-X02-v1','seed':SEED,'groups':{'train':42,'dev':15,'diagnostic':15},'positive_per_group':2,'negative_per_group':4,'top_k':K,'pairwise_hard':8,'pairwise_random':8,'C':1.0,'risk_rule':'max dev coverage at zero empirical wrong acceptances; else abstain all','status':'PROVISIONAL_SYNTHETIC_NOT_INDEPENDENT','candidate_tiers':['scope_only','official_current_scoped'],'no_external_data':True}
def dump(name,obj):
    (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n',encoding='utf-8')
def digest(p):return hashlib.sha256(p.read_bytes()).hexdigest()
dump('protocol.json',PROTOCOL)
print('CPU only |',platform.python_version(),np.__version__,sklearn.__version__)

## 1 — Corpus novo, rótulos separados
72 grupos de entidades, com 6 consultas por grupo. Cada grupo inclui duas positivas (PT/EN), quatro negativas (atributo ausente, referência inexistente, fonte exclusivamente externa e registo expirado). As fontes incluem atributos alternativos, textos semelhantes, histórico e claims. Os aliases e IDs são texto observável, nunca a chave gold. As formulações de diagnóstico são diferentes das de treino, mas são desenhadas no mesmo laboratório.

In [ ]:
# Fixed bilingual vocabulary, designed prior to any scoring. Not learned from gold.
ATTR={
 'owner':('responsável','owner','gestor','custodian','responsibility','supervisão'),
 'status':('estado','status','fase','stage','situação','state'),
 'capacity':('capacidade','capacity','limite','limit','lotação','quota'),
 'location':('localização','location','sede','site','local','venue'),
 'platform':('plataforma','platform','tecnologia','technology','sistema','stack'),
 'budget':('orçamento','budget','verba','allocation','custo','funding'),
 'deadline':('prazo','deadline','data limite','due date','vencimento','due'),
 'channel':('canal','channel','contacto','contact','rota','route'),
}
props=list(ATTR)
docs=[]; queries=[]; gold={}; groups={}; used=set()
def uid(prefix):
    while True:
        x=prefix+f'{rng.getrandbits(40):010X}'
        if x not in used: used.add(x);return x

def add_doc(split,scope,auth,start,end,text):
    d={'id':uid('D-'),'split':split,'scope':scope,'authority':auth,'from':start,'until':end,'text':text}
    docs.append(d);return d['id']
def add_query(split,group,lang,text,kind,target=None):
    q={'id':uid('Q-'),'split':split,'scope':'LAB-A','as_of':20261124,'language':lang,'text':text}
    queries.append(q);gold[q['id']]={'source_ids':[target] if target else [],'kind':kind,'group':group}
for split,n_groups in PROTOCOL['groups'].items():
    for i in range(n_groups):
        group=uid('G-'); ref=uid('RW-'); foreign=uid('RW-'); missing=uid('RW-')
        alias='Unidade '+uid('AX-'); groups[group]=split
        p1,p2,p3,p4,p5=rng.sample(props,5)
        # Each property has different literal terms on purpose. Cross-mentions create hard negatives.
        def term(p,j):return ATTR[p][j]
        def official(p,j,value):
            if i%3==0:return f'{ref} | {alias}. {term(p,j)}: {value}. Referência de auditoria a {term(p2 if p==p1 else p1,0)} sem atualização.'
            if i%3==1:return f'Registo {alias} ({ref}): {term(p,j)} confirmado como {value}. Histórico de acompanhamento em curso.'
            return f'{term(p,j)} = {value}; código {ref}; {alias}. Arquivo de operação e controlo.'
        a=add_doc(split,'LAB-A','OFFICIAL',20261001,None,official(p1,0,uid('VAL-')))
        b=add_doc(split,'LAB-A','OFFICIAL',20261001,None,official(p2,1,uid('VAL-')))
        add_doc(split,'LAB-A','OFFICIAL',20261001,None,official(p3,2,uid('VAL-')))
        add_doc(split,'LAB-A','OFFICIAL',20260701,20261001,official(p4,0,uid('OLD-')))
        add_doc(split,'LAB-A','UNVERIFIED',20261001,None,f'{ref} {alias}: {term(p1,1)} reportedly changed to {uid("CLAIM-")}.')
        add_doc(split,'LAB-B','OFFICIAL',20261001,None,f'{foreign}: {term(p1,0)} = {uid("VAL-")}. Referência externa.')
        add_doc(split,'LAB-B','OFFICIAL',20261001,None,f'{ref}: {term(p2,1)} = {uid("VAL-")}. Registo de outro projecto.')
        # Official but does not assert the requested p5 value; lexical bait.
        add_doc(split,'LAB-A','OFFICIAL',20261001,None,f'{ref} {alias}: referência a {term(p5,0)}; nenhum valor confirmado. Revisão administrativa.')
        # Surface templates are disjoint by split; entity names and IDs are random.
        def phrase(p,lang,entity):
            if split=='train':
                return f'Qual é o {term(p,0)} de {entity}?' if lang=='PT' else f'What is the {term(p,1)} for {entity}?'
            if split=='dev':
                return f'Indique {term(p,2)} relativo a {entity}.' if lang=='PT' else f'Find the recorded {term(p,3)} of {entity}.'
            return f'Para {entity}, preciso confirmar {term(p,4)}.' if lang=='PT' else f'For {entity}, retrieve the documented {term(p,5)}.'
        # Alternating literal ID and alias-only requests; entity string appears in public text.
        add_query(split,group,'PT',phrase(p1,'PT',ref),'answerable',a)
        add_query(split,group,'EN',phrase(p2,'EN',alias if i%2 else ref),'answerable',b)
        add_query(split,group,'PT',phrase(p5,'PT',ref),'unknown_attribute')
        add_query(split,group,'EN',phrase(p2,'EN',missing),'unknown_entity')
        add_query(split,group,'PT',phrase(p1,'PT',foreign),'foreign_only')
        add_query(split,group,'EN',phrase(p4,'EN',alias if i%2 else ref),'expired_only')
# IDs/ordering do not encode labels. No group or kind in public query/doc records.
rng.shuffle(docs);rng.shuffle(queries)
assert len(docs)==576 and len(queries)==432 and len(groups)==72
assert len({d['id'] for d in docs})==len(docs) and len({q['id'] for q in queries})==len(queries)
byid={d['id']:d for d in docs}
for q in queries:
    for sid in gold[q['id']]['source_ids']:
        d=byid[sid]; assert d['split']==q['split'] and d['scope']==q['scope'] and d['authority']=='OFFICIAL'
dump('documents.json',docs);dump('queries.json',queries);dump('rule_gold.json',gold)
print('Documents',len(docs),'Queries',len(queries),'Groups',Counter(groups.values()))

## 2 — Features e duas populações de candidatos
Pesquisa apenas no split e âmbito autorizados; sem filtro pela entidade gold. A comparação primária aplica a mesma elegibilidade oficial e temporal a todos os métodos. A comparação lexical de recuperação simples é uma linha de controlo separada. TF-IDF, BM25 IDF e LSA são ajustados somente nas fontes autorizadas de treino (LSA não é um modelo semântico pré-treinado).

In [ ]:
def tokens(t):return re.findall(r'[^\W_]+',t.casefold(),flags=re.UNICODE)
def ids(t):return set(re.findall(r'\b(?:RW|AX)-[0-9A-F]{10}\b',t.upper()))
def concepts(t):
    ts=' '+ ' '.join(tokens(t))+' '
    return {k for k,terms in ATTR.items() if any(' '+ ' '.join(tokens(x))+' ' in ts for x in terms)}
def eligible(q,d):
    return d['authority']=='OFFICIAL' and d['from']<=q['as_of'] and (d['until'] is None or q['as_of']<d['until'])
train_docs=[d for d in docs if d['split']=='train' and d['scope']=='LAB-A']
train_text=[d['text'] for d in train_docs]
vec=TfidfVectorizer(analyzer='char',ngram_range=(2,4),min_df=2,sublinear_tf=True,dtype=np.float64).fit(train_text)
word=TfidfVectorizer(ngram_range=(1,2),min_df=1,sublinear_tf=True).fit(train_text)
lsa=TruncatedSVD(n_components=32,random_state=SEED).fit(word.transform(train_text))
N=len(train_docs);df=Counter(w for d in train_docs for w in set(tokens(d['text'])))
avgdl=np.mean([len(tokens(d['text'])) for d in train_docs]); default_idf=math.log(1+(N+.5)/.5)

def bm25(q,dt):
    c=Counter(dt); dl=len(dt); k1=1.2;b=.75
    return sum(math.log(1+(N-df[w]+.5)/(df[w]+.5))*c[w]*(k1+1)/(c[w]+k1*(1-b+b*dl/avgdl)) for w in set(tokens(q)) if c[w])
def norm(x):
    m=max(float(np.max(x)),1e-12);return np.maximum(x,0)/m
all_text=[d['text'] for d in docs]; mat=vec.transform(all_text); latent=lsa.transform(word.transform(all_text))
ix={d['id']:i for i,d in enumerate(docs)}
features=['bm25_relative','char_tfidf','lsa_cos','token_overlap','exact_ref_match','property_overlap','property_coverage','negation_notice','bm25_rank_fraction']
cache={}
for q in queries:
    pool=[d for d in docs if d['split']==q['split'] and d['scope']==q['scope']]
    inds=[ix[d['id']] for d in pool]; text=q['text']; qt=set(tokens(text)); qc=concepts(text); qi=ids(text)
    bm=np.array([bm25(text,tokens(d['text'])) for d in pool]); ch=(mat[inds]@vec.transform([text]).T).toarray().ravel()
    z=lsa.transform(word.transform([text]))[0]; l=latent[inds]; cs=(l@z)/np.maximum(np.linalg.norm(l,axis=1)*np.linalg.norm(z),1e-12)
    rank=np.argsort(np.argsort(-bm,kind='stable'),kind='stable')
    X=np.column_stack([norm(bm),ch,cs,[len(qt&set(tokens(d['text'])))/max(1,len(qt)) for d in pool],[float(bool(qi&ids(d['text']))) for d in pool],[float(bool(qc&concepts(d['text']))) for d in pool],[len(qc&concepts(d['text']))/max(1,len(qc)) for d in pool],[float('nenhum valor confirmado' in d['text'].casefold()) for d in pool],1-rank/max(1,len(pool)-1)])
    keep=np.array([eligible(q,d) for d in pool],dtype=bool)
    assert np.all(np.isfinite(X)) and keep.any()
    cache[q['id']]={'ids':[d['id'] for d in pool],'X':X,'eligible':keep,'bm25':bm}
    assert set(gold[q['id']]['source_ids'])<=set(np.array(cache[q['id']]['ids'])[keep])
print('Feature extraction finished; same eligible pool for all primary methods.')

## 3 — Treino comparativo
Controlo pointwise treinado de novo em X02, com pesos equilibrados por consulta/classe. Pairwise aprende a preferir a fonte positiva a 8 negativos difíceis e 8 aleatórios, por consulta, com diferenças simétricas. Dois modelos pairwise (texto e features completas). Negativas sem gold não entram no treino do ranker; entram no desenvolvimento da decisão. Nenhum ajuste com gold de diagnóstico.

In [ ]:
train=[q for q in queries if q['split']=='train' and gold[q['id']]['source_ids']]
# Train-only feature scaling, applied consistently to difference vectors.
train_X=[]; train_y=[]; train_w=[]; pair_X=[]; pair_y=[]; pair_records=[]
for q in train:
    c=cache[q['id']]; X=c['X'][c['eligible']]; did=np.array(c['ids'])[c['eligible']]; y=(did==gold[q['id']]['source_ids'][0]).astype(int)
    pos=int(np.flatnonzero(y)[0]); neg=np.flatnonzero(y==0)
    train_X.extend(X);train_y.extend(y);train_w.extend(np.where(y==1,.5,.5/len(neg)))
    heuristic=.45*X[:,0]+.25*X[:,1]+.30*X[:,4]+.20*X[:,6]-.15*X[:,7]
    hard=sorted(neg,key=lambda i:(-heuristic[i],did[i]))[:PROTOCOL['pairwise_hard']]
    rest=[i for i in neg if i not in hard]; sampled=rng.sample(rest,min(len(rest),PROTOCOL['pairwise_random']))
    for j in hard+sampled:
        delta=X[pos]-X[j];pair_X.extend([delta,-delta]);pair_y.extend([1,0]);pair_records.append({'query_id':q['id'],'positive':str(did[pos]),'negative':str(did[j]),'selection':'hard' if j in hard else 'random'})
TX=np.array(train_X); TY=np.array(train_y);TW=np.array(train_w)*len(TY)/sum(train_w)
scale=StandardScaler().fit(TX)
PX=np.array(pair_X)/scale.scale_; PY=np.array(pair_y)
text_idx=[0,1,2,3,8]; full_idx=list(range(len(features)))
point=LogisticRegression(C=1.0,max_iter=1000,random_state=SEED).fit(scale.transform(TX),TY,sample_weight=TW)
pair_text=LogisticRegression(C=1.0,fit_intercept=False,max_iter=1000,random_state=SEED).fit(PX[:,text_idx],PY)
pair_full=LogisticRegression(C=1.0,fit_intercept=False,max_iter=1000,random_state=SEED).fit(PX,PY)
# Ablation removes explicit identifier feature but retains property signals.
no_id_idx=[i for i in full_idx if i!=4]
pair_no_id=LogisticRegression(C=1.0,fit_intercept=False,max_iter=1000,random_state=SEED).fit(PX[:,no_id_idx],PY)
models={'Pointwise':(point,full_idx),'PairwiseText':(pair_text,text_idx),'RW-R1-X02':(pair_full,full_idx),'PairwiseNoID':(pair_no_id,no_id_idx)}
assert len(train)==84 and len(pair_records)==84*16 and len(PY)==2*len(pair_records)
assert all(next(q for q in queries if q['id']==r['query_id'])['split']=='train' for r in pair_records)
dump('training_pairs.json',pair_records)
dump('weights.json',{'features':features,'scale':scale.scale_.tolist(),'mean':scale.mean_.tolist(),'models':{k:{'coef':m.coef_.ravel().tolist(),'intercept':m.intercept_.tolist(),'feature_indices':idx} for k,(m,idx) in models.items()},'pointwise_rows':len(TY),'positive_rows':int(TY.sum()),'pairwise_directions':len(PY),'note':'diagnostic parameters; no production inference package'})
print('Pointwise rows:',len(TY),'positive:',TY.sum(),'Pairwise comparisons:',len(pair_records),'directed rows:',len(PY))

## 4 — Rankings e avaliação
Gold só é usado na aprendizagem de train e no cálculo das métricas, não na selecção de candidatos. Ranking primário: BM25, CharTFIDF, LSA32, Hybrid, Hybrid-ID-Property (regra explícita forte), Pointwise, PairwiseText, PairwiseNoID, RW-R1-X02. A linha BM25-Scope mede recuperação sem filtro de autoridade/tempo. Negativas: cobertura de candidatos, não acerto de resposta.

In [ ]:
rows=[]; top_features={}
for q in queries:
    c=cache[q['id']];X=c['X']; eligible_mask=c['eligible']; did=np.array(c['ids']); z=scale.transform(X)
    t0=time.perf_counter()
    hybrid=.50*X[:,0]+.30*X[:,1]+.20*np.maximum(X[:,2],0)
    explicit=hybrid+.8*X[:,4]+.45*X[:,6]-.65*X[:,7]
    scores={'BM25-Scope':(c['bm25'],np.ones(len(did),dtype=bool)),'BM25':(c['bm25'],eligible_mask),'CharTFIDF':(X[:,1],eligible_mask),'LSA32':(X[:,2],eligible_mask),'Hybrid':(hybrid,eligible_mask),'Hybrid-ID-Property':(explicit,eligible_mask),'Pointwise':(point.decision_function(z),eligible_mask),'PairwiseText':(pair_text.decision_function(z[:,text_idx]),eligible_mask),'PairwiseNoID':(pair_no_id.decision_function(z[:,no_id_idx]),eligible_mask),'RW-R1-X02':(pair_full.decision_function(z),eligible_mask)}
    for method,(v,mask) in scores.items():
        order=sorted(np.flatnonzero(mask),key=lambda i:(-float(v[i]),str(did[i])))[:K]
        assert all(byid[str(did[i])]['scope']==q['scope'] and byid[str(did[i])]['split']==q['split'] for i in order)
        if method!='BM25-Scope':assert all(eligible(q,byid[str(did[i])]) for i in order)
        rows.append({'query_id':q['id'],'method':method,'tier':'scope_only' if method=='BM25-Scope' else 'official_current_scoped','candidate_count':int(mask.sum()),'source_ids':[str(did[i]) for i in order],'scores':[float(v[i]) for i in order]})
        if method=='RW-R1-X02':
            best=order[0];margin=float(v[best]-v[order[1]]) if len(order)>1 else 0.
            # Fixed decision score, no probability claim: explicit matching + rank margin.
            # Negative statements reduce confidence; no positive coverage forced.
            confidence=float(X[best,4]*X[best,6]*(1-X[best,7])*(.5+.5*np.tanh(max(0,margin))))
            top_features[q['id']]={'source_id':str(did[best]),'confidence':confidence,'margin':margin,'identity':float(X[best,4]),'property':float(X[best,6]),'negative_notice':float(X[best,7])}
    # Whole query timings not comparable per method, feature extraction excluded.
assert len(rows)==432*10
qmap={q['id']:q for q in queries}; rmap={(r['query_id'],r['method']):r for r in rows};assert len(rmap)==len(rows)
methods=list(scores)
def rank_metrics(qids,method):
    vals=[]
    for qid in qids:
        g=gold[qid]['source_ids'];r=rmap[qid,method];
        if not g:continue
        ranks=[i+1 for i,s in enumerate(r['source_ids']) if s in g];k=min(ranks,default=0)
        vals.append([float(k==1),float(0<k<=2),float(0<k<=5),1/k if k else 0,1/math.log2(k+1) if k else 0])
    mean=np.mean(vals,axis=0).tolist() if vals else [None]*5
    return dict(zip(['hit1','hit2','hit5','mrr5','ndcg5'],mean))|{'n_positive':len(vals),'n_queries':len(qids),'negative_with_candidates':sum(not gold[x]['source_ids'] and bool(rmap[x,method]['source_ids']) for x in qids)}
metrics=[]
for split in PROTOCOL['groups']:
    for lang in ['ALL','PT','EN']:
        qids=[q['id'] for q in queries if q['split']==split and (lang=='ALL' or q['language']==lang)]
        for method in methods:metrics.append({'split':split,'language':lang,'method':method}|rank_metrics(qids,method))
dump('raw_rankings.json',rows);dump('metrics.json',metrics);dump('decision_inputs.json',top_features)
diag=[x for x in metrics if x['split']=='diagnostic' and x['language']=='ALL']
for x in diag: print(f"{x['method']:21s} hit1={x['hit1']:.3f} ndcg5={x['ndcg5']:.3f} n={x['n_positive']}")

## 5 — Confiança e efeito pareado
Score de confiança fixo (não probabilidade calibrada), calculado com margem, entidade, atributo e aviso de ausência. Limiar escolhido exclusivamente no dev: maior cobertura com zero erros observados, ou rejeição total. Reportar risco indefinido quando não houver aceites. Bootstrap por grupo, 2.000 reamostragens, apenas como descrição da variabilidade sintética; não inferência populacional independente.

In [ ]:
def outcome(split,threshold):
    qids=[q['id'] for q in queries if q['split']==split]
    accepted=[i for i in qids if top_features[i]['confidence']>=threshold]
    bad=sum(top_features[i]['source_id'] not in gold[i]['source_ids'] for i in accepted)
    neg={k:{'n':sum(gold[i]['kind']==k for i in qids),'accepted':sum(gold[i]['kind']==k for i in accepted)} for k in sorted({gold[i]['kind'] for i in qids}) if k!='answerable'}
    return {'split':split,'threshold':float(threshold),'n':len(qids),'accepted':len(accepted),'wrong':bad,'coverage':len(accepted)/len(qids),'risk':bad/len(accepted) if accepted else None,'negative':neg}
dev_grid=[outcome('dev',t) for t in list(np.linspace(0,1,101))+[1.01]]
valid=[r for r in dev_grid if r['wrong']==0 and r['accepted']>0]
chosen=max(valid,key=lambda r:(r['coverage'],r['threshold'])) if valid else dev_grid[-1]
decision={'rule':PROTOCOL['risk_rule'],'dev_grid':dev_grid,'selected_dev':chosen,'diagnostic':outcome('diagnostic',chosen['threshold'])}
dump('abstention.json',decision)
# Paired group effect of full model against explicit baseline on the same eligible corpus.
diffs=[]
for g in sorted(k for k,v in groups.items() if v=='diagnostic'):
    qi=[q['id'] for q in queries if gold[q['id']]['group']==g]
    diffs.append(rank_metrics(qi,'RW-R1-X02')['ndcg5']-rank_metrics(qi,'Hybrid-ID-Property')['ndcg5'])
arr=np.array(diffs); resamples=np_rng.choice(arr,size=(2000,len(arr)),replace=True).mean(axis=1)
paired={'comparison':'RW-R1-X02 minus Hybrid-ID-Property','diagnostic_groups':len(arr),'mean_delta':float(arr.mean()),'bootstrap_percentiles_2_5_97_5':np.quantile(resamples,[.025,.975]).tolist(),'per_group_delta':diffs,'caveat':'synthetic authored groups; not independent population inference'}
dump('paired_effect.json',paired)
print('Dev selected:',chosen['threshold'],chosen['accepted'],'/',chosen['n'],'wrong',chosen['wrong'])
print('Diagnostic:',decision['diagnostic']);print('Paired effect:',paired['mean_delta'],paired['bootstrap_percentiles_2_5_97_5'])

## 6 — Resultados visuais e pacote de evidências
O modelo só pode ser considerado candidato a nova investigação se o ganho sobre a regra explícita for sustentado; esta experiência não autoriza deployment. O ZIP guarda ranking bruto, métricas, pesos, pares, parâmetros, dados, ambiente e hashes, para recontagem independente. Não existe oracle reservado neste notebook.

In [ ]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(figsize=(9,5.5)); primary=[x for x in diag if x['method']!='BM25-Scope']; primary.sort(key=lambda x:x['ndcg5'])
ax.barh([x['method'] for x in primary],[x['ndcg5'] for x in primary]);ax.set_xlim(0,1.08);ax.set_xlabel('nDCG@5 (30 positive diagnostic queries)');ax.set_title('E1-X02 — Equal official/current candidate pool')
for i,x in enumerate(primary):ax.text(x['ndcg5']+.012,i,f"{x['ndcg5']:.3f}",va='center')
fig.tight_layout();fig.savefig(OUT/'retrieval.png',dpi=130,bbox_inches='tight');plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4));rr=[x for x in dev_grid if x['risk'] is not None];ax.scatter([x['coverage'] for x in rr],[x['risk'] for x in rr]);ax.set(xlim=(-.03,1.03),ylim=(-.03,1.03),xlabel='Coverage',ylabel='Wrong / accepted',title='Dev only — risk versus coverage (n=90)');ax.grid(alpha=.2);fig.tight_layout();fig.savefig(OUT/'risk_coverage.png',dpi=130);plt.show();plt.close(fig)
# Check rankings and derived metrics before archive; no assertion requiring high scores.
assert all(len(r['source_ids'])==K and len(set(r['source_ids']))==K and all(a>=b for a,b in zip(r['scores'],r['scores'][1:])) for r in rows)
assert len(metrics)==90 and len(dev_grid)==102
assert chosen['split']=='dev' and chosen['wrong']==0
assert len([q for q in queries if q['split']=='diagnostic'])==90
summary={'classification':'PROVISIONAL_SYNTHETIC_NOT_INDEPENDENT','e1_data_ready':'REVIEW_REQUIRED','human_reviews':0,'provider_comparison':'NOT_RUN','scope_violations':0,'documents':len(docs),'queries':len(queries),'rankings':len(rows),'pairs':len(pair_records),'metrics':len(metrics),'diagnostic':diag,'abstention':decision['diagnostic'],'paired_effect':paired,'environment':{'python':platform.python_version(),'numpy':np.__version__,'sklearn':sklearn.__version__,'platform':platform.platform(),'utc':datetime.now(timezone.utc).isoformat()},'limitations':['same-author synthetic rule labels','no independent holdout','no answer generation or real authorization','confidence not calibrated probability','no pretrained dense or vendor comparison','historical X01 comparison is not matched-corpus evidence']}
dump('summary.json',summary)
files={p.name:{'sha256':digest(p),'bytes':p.stat().st_size} for p in sorted(OUT.iterdir()) if p.is_file()};dump('MANIFEST.json',{'schema':'RIGHTWARE_SIC_E1_X02_EVIDENCE_V1','files':files})
archive=ROOT/'RIGHTWARE_SIC_E1_X02_EVIDENCE.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):z.write(p,p.name)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    manifest=json.loads(z.read('MANIFEST.json'));assert set(z.namelist())==set(manifest['files'])|{'MANIFEST.json'}
    for name,meta in manifest['files'].items():assert hashlib.sha256(z.read(name)).hexdigest()==meta['sha256'] and len(z.read(name))==meta['bytes']
print('E1_X02_EXECUTED_PROVISIONAL | manifest entries:',len(files),'| sha256:',digest(archive))
print('E1_DATA_READY=REVIEW_REQUIRED | no scientific approval')
display(FileLink(archive.name))